# Deep SARSA UCB-VAE — Top-5 validation trên HPG BAD (20 seeds)

Notebook phân phối: mỗi tài khoản Kaggle chọn đúng một `CONFIG_ID` từ 1 đến 5.
Kết quả được append ngay sau mỗi seed và tự động resume nếu session bị ngắt.


In [ ]:
!git clone https://github.com/kohi-vip/SARSA_FinancialRL.git

In [ ]:
!pip install numpy pandas matplotlib tqdm torch TA-Lib optuna

In [ ]:
# CODE 1 — KAGGLE ACCOUNT CONFIGURATION
# Trên năm tài khoản, chỉ đổi CONFIG_ID lần lượt thành 1, 2, 3, 4, 5.
CONFIG_ID = 1
RUN_VALIDATION = True

# Global Top-5 lấy từ top5_ucb_configs.csv sau Grid Search 48 cấu hình.
TOP5_UCB_CONFIGS = {
    1: {"ucb_config_id": "UCB-20", "gamma": 0.95, "beta": 0.15, "beta_decay": 0.94, "delta": 1.0, "beta_min": 0.01},
    2: {"ucb_config_id": "UCB-30", "gamma": 0.95, "beta": 0.30, "beta_decay": 0.91, "delta": 2.0, "beta_min": 0.01},
    3: {"ucb_config_id": "UCB-40", "gamma": 0.95, "beta": 0.85, "beta_decay": 0.91, "delta": 0.5, "beta_min": 0.01},
    4: {"ucb_config_id": "UCB-16", "gamma": 0.95, "beta": 0.15, "beta_decay": 0.91, "delta": 0.5, "beta_min": 0.01},
    5: {"ucb_config_id": "UCB-11", "gamma": 0.95, "beta": 0.03, "beta_decay": 0.97, "delta": 1.0, "beta_min": 0.01},
}

if CONFIG_ID not in TOP5_UCB_CONFIGS:
    raise ValueError("CONFIG_ID phải là một trong [1, 2, 3, 4, 5].")

UCB_CONFIG = dict(TOP5_UCB_CONFIGS[CONFIG_ID])
gamma = UCB_CONFIG["gamma"]
beta = UCB_CONFIG["beta"]
beta_decay = UCB_CONFIG["beta_decay"]
delta = UCB_CONFIG["delta"]
beta_min = UCB_CONFIG["beta_min"]

print(f"Account CONFIG_ID={CONFIG_ID}: {UCB_CONFIG}")


In [ ]:
# CODE 2 — 20 VALIDATION SEEDS AND SYNCHRONIZED SEEDING
import gc
import json
import math
import os
import random
import subprocess
import sys
import time
import traceback
from pathlib import Path
from typing import Any, Dict, Mapping, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

VALIDATION_SEEDS = list(range(42, 62))  # Dung dung 20 seed: 42..61.


def set_seed(seed_value: int) -> None:
    # Đồng bộ seed Python, NumPy, PyTorch CPU và mọi CUDA device.
    seed_value = int(seed_value)
    os.environ["PYTHONHASHSEED"] = str(seed_value)
    random.seed(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    try:
        torch.use_deterministic_algorithms(True, warn_only=True)
    except TypeError:
        torch.use_deterministic_algorithms(True)


set_seed(VALIDATION_SEEDS[0])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("DEVICE =", DEVICE)
print("VALIDATION_SEEDS =", VALIDATION_SEEDS)


In [ ]:
# CODE 3 — INCREMENTAL LOGGING AND AUTO-RESUME
KAGGLE_WORKING = Path("/kaggle/working")
OUTPUT_DIR = KAGGLE_WORKING if KAGGLE_WORKING.exists() else Path.cwd() / "kaggle_working"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RESULTS_CSV = OUTPUT_DIR / f"validation_results_config_{CONFIG_ID}.csv"
ERRORS_CSV = OUTPUT_DIR / f"validation_errors_config_{CONFIG_ID}.csv"
SUMMARY_CSV = OUTPUT_DIR / f"validation_summary_config_{CONFIG_ID}.csv"


def append_dict_to_csv(csv_path: Path, result_dict: Mapping[str, Any]) -> None:
    csv_path.parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame([dict(result_dict)]).to_csv(
        csv_path,
        mode="a",
        header=not csv_path.exists(),
        index=False,
    )


def completed_seeds() -> set[int]:
    if not RESULTS_CSV.exists():
        return set()
    try:
        frame = pd.read_csv(RESULTS_CSV)
    except pd.errors.EmptyDataError:
        return set()
    if "seed" not in frame.columns:
        raise ValueError(f"File resume thiếu cột seed: {RESULTS_CSV}")

    # Không resume nhầm nếu người dùng đổi CONFIG_ID/hyperparameters nhưng giữ CSV cũ.
    expected = {
        "account_config_id": CONFIG_ID,
        "ucb_config_id": UCB_CONFIG["ucb_config_id"],
        "gamma": gamma,
        "beta": beta,
        "beta_decay": beta_decay,
        "delta": delta,
        "beta_min": beta_min,
    }
    for column, expected_value in expected.items():
        if column not in frame.columns:
            raise ValueError(f"File resume thiếu cột cấu hình {column}: {RESULTS_CSV}")
        values = frame[column].dropna().unique()
        if isinstance(expected_value, (int, float)) and not isinstance(expected_value, bool):
            if any(not np.isclose(float(value), float(expected_value)) for value in values):
                raise ValueError(f"CSV cũ không khớp {column}={expected_value}: {values}")
        elif any(str(value) != str(expected_value) for value in values):
            raise ValueError(f"CSV cũ không khớp {column}={expected_value}: {values}")
    return {int(seed) for seed in frame["seed"].dropna().tolist() if int(seed) in VALIDATION_SEEDS}


def seeds_remaining() -> list[int]:
    done = completed_seeds()
    remaining = [seed for seed in VALIDATION_SEEDS if seed not in done]
    print(f"Completed seeds: {sorted(done)}")
    print(f"Seeds remaining: {remaining}")
    return remaining


def append_result_to_csv(result_dict: Mapping[str, Any]) -> None:
    append_dict_to_csv(RESULTS_CSV, result_dict)


print("RESULTS_CSV =", RESULTS_CSV)


In [ ]:
# CODE 4 — PROJECT, MODEL, ENVIRONMENT, SHARED BACKBONE AND LOCKED VAE-09
import importlib.util


def find_or_clone_project() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd / "SARSA_FinancialRL", Path("/kaggle/working/SARSA_FinancialRL")]
    candidates.extend(cwd.parents)
    for candidate in candidates:
        if (candidate / "data").exists() and (candidate / "environments").exists():
            return candidate

    destination = Path("/kaggle/working/SARSA_FinancialRL")
    if not destination.exists():
        subprocess.run(
            ["git", "clone", "https://github.com/kohi-vip/SARSA_FinancialRL.git", str(destination)],
            check=True,
        )
    return destination


PROJECT_ROOT = find_or_clone_project()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

def load_ucb_source(project_root: Path):
    source_candidates = [
        project_root / "application" / "EIDT_Project" / "UCB_2_pro_fixed.py",
        project_root / "application" / "EIDT_Project" / "UCB_2.py",
    ]
    source_path = next((path for path in source_candidates if path.exists()), None)
    if source_path is None:
        raise FileNotFoundError(f"Không tìm thấy UCB source trong: {source_candidates}")
    spec = importlib.util.spec_from_file_location("ucb_validation_source", source_path)
    if spec is None or spec.loader is None:
        raise ImportError(f"Không thể tạo import spec cho {source_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = module
    spec.loader.exec_module(module)
    print("Loaded UCB source:", source_path)
    return module


ucb_source = load_ucb_source(PROJECT_ROOT)
if ucb_source.stockMDP is None:
    from environments.stock_trading_env.mdp import StockTradingMDP
    ucb_source.stockMDP = StockTradingMDP


def load_hpg_bad_data(project_root: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    # Dùng CSV đã tính MACD/RSI/CCI/ADX, không phụ thuộc TA-Lib trên Kaggle.
    data_root = project_root / "data" / "data_storer" / "data_research"
    train_path = data_root / "train" / "bad_train_HPG.csv"
    test_path = data_root / "test" / "bad_test_HPG.csv"
    if not train_path.exists() or not test_path.exists():
        raise FileNotFoundError(f"Thiếu HPG BAD CSV: {train_path}, {test_path}")
    train_frame = pd.read_csv(train_path)
    test_frame = pd.read_csv(test_path)
    required = {"time", "close", "MACD", "RSI", "CCI", "ADX"}
    for name, frame in (("train", train_frame), ("test", test_frame)):
        missing = sorted(required - set(frame.columns))
        if missing:
            raise ValueError(f"HPG BAD {name} thiếu cột: {missing}")
        frame["time"] = pd.to_datetime(frame["time"], errors="raise")
    return train_frame.reset_index(drop=True), test_frame.reset_index(drop=True)

SHARED_BACKBONE = {
    "episodes": 45,
    "gamma": gamma,
    "smoothing_alpha": 0.60,
    "nn_epochs": 1,
    "nn_lr": 5e-5,
    "q_batch_size": 128,
    "balance_init": 1000,
    "k": 5,
    "min_balance": -100,
}

LOCKED_VAE09 = {
    "vae_lr": 0.001,
    "vae_beta_kl": 0.01,
    "vae_latent_dim": 16,
    "vae_epochs": 30,
    "vae_batch_size": 256,
    "bootstrap_trajectories": 5,
    "bootstrap_vae_updates": 100,
    "vae_updates_per_q_patch": 1,
    "vae_replay_capacity": 50_000,
}

RUN_CONFIG = {
    **SHARED_BACKBONE,
    **LOCKED_VAE09,
    **UCB_CONFIG,
    "lam": 1.0,
}

# QsaUCB imported from source is exactly MLP 7 -> 32 -> 11.
q_architecture_check = ucb_source.QsaUCB(input_size=7, num_classes=11)
print("Q-network =", q_architecture_check)
del q_architecture_check
print("RUN_CONFIG =", json.dumps(RUN_CONFIG, indent=2))


def run_single_validation_seed(
    train_series: pd.DataFrame,
    test_series: pd.DataFrame,
) -> Dict[str, float]:
    mdp = ucb_source.stockMDP(
        balance_init=RUN_CONFIG["balance_init"],
        k=RUN_CONFIG["k"],
        min_balance=RUN_CONFIG["min_balance"],
    )

    vae, vae_optimizer, scaler, vae_buffer, bootstrap_losses = (
        ucb_source.initialize_joint_vae_components(
            train_series=train_series,
            mdp=mdp,
            latent_dim=RUN_CONFIG["vae_latent_dim"],
            vae_lr=RUN_CONFIG["vae_lr"],
            replay_capacity=RUN_CONFIG["vae_replay_capacity"],
            bootstrap_trajectories=RUN_CONFIG["bootstrap_trajectories"],
            bootstrap_vae_updates=RUN_CONFIG["bootstrap_vae_updates"],
            vae_batch_size=RUN_CONFIG["vae_batch_size"],
            vae_beta_kl=RUN_CONFIG["vae_beta_kl"],
            verbose=False,
        )
    )

    # Thực thi đúng vae_epochs=30 trên bootstrap replay buffer.
    vae_steps_per_epoch = max(1, math.ceil(len(vae_buffer) / RUN_CONFIG["vae_batch_size"]))
    vae_pretrain_losses = []
    for _epoch in range(RUN_CONFIG["vae_epochs"]):
        for _step in range(vae_steps_per_epoch):
            stats = ucb_source.update_vae_from_replay(
                vae,
                vae_optimizer,
                vae_buffer,
                scaler,
                mdp,
                batch_size=RUN_CONFIG["vae_batch_size"],
                vae_beta_kl=RUN_CONFIG["vae_beta_kl"],
            )
            if stats is not None:
                vae_pretrain_losses.append(float(stats["loss"]))

    policy, q_network, learning_curve, online_vae_losses = (
        ucb_source.train_deep_sarsa_ucb_vae(
            mdp=mdp,
            train_series=train_series,
            test_series=test_series,
            vae=vae,
            vae_optimizer=vae_optimizer,
            vae_buffer=vae_buffer,
            scaler=scaler,
            episodes=RUN_CONFIG["episodes"],
            gamma=RUN_CONFIG["gamma"],
            alpha=RUN_CONFIG["smoothing_alpha"],
            nn_epochs=RUN_CONFIG["nn_epochs"],
            nn_lr=RUN_CONFIG["nn_lr"],
            beta=RUN_CONFIG["beta"],
            delta=RUN_CONFIG["delta"],
            lam=RUN_CONFIG["lam"],
            batch_size=RUN_CONFIG["q_batch_size"],
            vae_batch_size=RUN_CONFIG["vae_batch_size"],
            vae_updates_per_q_batch=RUN_CONFIG["vae_updates_per_q_patch"],
            vae_beta_kl=RUN_CONFIG["vae_beta_kl"],
            beta_decay=RUN_CONFIG["beta_decay"],
            beta_min=RUN_CONFIG["beta_min"],
            verbose=False,  # Tat episode tqdm de output Kaggle khong bi tran dong.
        )
    )

    final_profit = float(
        mdp.interact_test(
            policy,
            train_series=train_series,
            test_series=test_series,
            series_name="test",
            verbose=False,
        )
    )
    portfolio, _states, _actions = ucb_source.collect_portfolio_history(
        mdp, policy, train_series, test_series
    )
    final_portfolio = float(portfolio[-1]) if len(portfolio) else float(mdp.balance_init + final_profit)
    arr_percent, _roi_percent = ucb_source.agent_annual_return(
        float(mdp.balance_init),
        final_portfolio,
        test_series.iloc[0]["time"],
        test_series.iloc[-1]["time"],
    )

    return {
        "profit": final_profit,
        "arr_percent": float(arr_percent),
        "volatility_percent": float(ucb_source.calculate_volatility(portfolio)),
        "sharpe_ratio": float(ucb_source.calculate_sharpe_ratio(portfolio)),
        "max_drawdown_percent": float(ucb_source.calculate_max_drawdown(portfolio)),
        "final_portfolio": final_portfolio,
        "final_beta": float(policy.beta),
        "bootstrap_vae_loss": float(np.mean(bootstrap_losses)) if bootstrap_losses else np.nan,
        "pretrain_vae_loss": float(np.mean(vae_pretrain_losses)) if vae_pretrain_losses else np.nan,
        "online_vae_loss": float(np.mean(online_vae_losses)) if online_vae_losses else np.nan,
        "final_learning_curve_profit": float(learning_curve[-1]) if learning_curve else np.nan,
    }


In [ ]:
# CODE 5 — MAIN 20-SEED TRAINING AND BAD-PERIOD VALIDATION LOOP
if RUN_VALIDATION:
    bad_train_hpg, bad_test_hpg = load_hpg_bad_data(PROJECT_ROOT)
    print({"bad_train": len(bad_train_hpg), "bad_test": len(bad_test_hpg)})

    seeds_to_run = seeds_remaining()
    completed_count = len(VALIDATION_SEEDS) - len(seeds_to_run)
    for seed in seeds_to_run:
        started_at = pd.Timestamp.now()
        started_clock = time.perf_counter()
        try:
            set_seed(seed)
            metrics = run_single_validation_seed(bad_train_hpg, bad_test_hpg)
            result_row = {
                "account_config_id": CONFIG_ID,
                "ucb_config_id": UCB_CONFIG["ucb_config_id"],
                "seed": int(seed),
                "gamma": gamma,
                "beta": beta,
                "beta_decay": beta_decay,
                "delta": delta,
                "beta_min": beta_min,
                **LOCKED_VAE09,
                **metrics,
                "elapsed_seconds": float(time.perf_counter() - started_clock),
                "started_at": started_at.isoformat(),
                "completed_at": pd.Timestamp.now().isoformat(),
            }
            append_result_to_csv(result_row)
            completed_count += 1
            percent = 100.0 * completed_count / len(VALIDATION_SEEDS)
            print(
                f"PROGRESS {completed_count:02d}/{len(VALIDATION_SEEDS)} ({percent:5.1f}%) | "
                f"seed={seed:02d} | profit={metrics['profit']:.6f} | "
                f"sharpe={metrics['sharpe_ratio']:.6f} | "
                f"elapsed={result_row['elapsed_seconds']:.1f}s",
                flush=True,
            )
        except Exception as error:
            error_row = {
                "account_config_id": CONFIG_ID,
                "ucb_config_id": UCB_CONFIG["ucb_config_id"],
                "seed": int(seed),
                "error_type": type(error).__name__,
                "error": str(error),
                "traceback": traceback.format_exc(),
                "failed_at": pd.Timestamp.now().isoformat(),
            }
            append_dict_to_csv(ERRORS_CSV, error_row)
            print(f"FAILED seed={seed:02d}: {type(error).__name__}: {error}", flush=True)
        finally:
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.synchronize()
                torch.cuda.empty_cache()
    print(f"Validation loop finished: {completed_count}/{len(VALIDATION_SEEDS)} successful seeds.")
else:
    print("RUN_VALIDATION=False: main loop was not executed.")


In [ ]:
# CODE 6 — FINAL MEAN ± STD STATISTICAL AGGREGATION
METRICS = {
    "Profit": "profit",
    "ARR (%)": "arr_percent",
    "Volatility (%)": "volatility_percent",
    "Sharpe Ratio": "sharpe_ratio",
    "Max Drawdown (%)": "max_drawdown_percent",
}

if not RESULTS_CSV.exists():
    print(f"Chưa có kết quả để tổng hợp: {RESULTS_CSV}")
else:
    validation_frame = pd.read_csv(RESULTS_CSV)
    validation_frame = validation_frame[validation_frame["seed"].isin(VALIDATION_SEEDS)].copy()
    if validation_frame.empty:
        raise ValueError(f"CSV chua co ket qua cho validation seeds {VALIDATION_SEEDS}.")
    validation_frame = (
        validation_frame.sort_values("completed_at")
        .drop_duplicates(subset=["seed"], keep="last")
        .sort_values("seed")
        .reset_index(drop=True)
    )
    missing_metrics = [column for column in METRICS.values() if column not in validation_frame]
    if missing_metrics:
        raise ValueError(f"CSV thiếu các metric: {missing_metrics}")

    summary_rows = []
    for label, column in METRICS.items():
        values = pd.to_numeric(validation_frame[column], errors="raise")
        mean_value = float(values.mean())
        std_value = float(values.std(ddof=1)) if len(values) > 1 else 0.0
        summary_rows.append(
            {
                "metric": label,
                "mean": mean_value,
                "std": std_value,
                "mean ± std": f"{mean_value:.6f} ± {std_value:.6f}",
            }
        )

    summary_frame = pd.DataFrame(summary_rows)
    summary_frame.insert(0, "ucb_config_id", UCB_CONFIG["ucb_config_id"])
    summary_frame.insert(0, "account_config_id", CONFIG_ID)
    summary_frame["seeds_completed"] = validation_frame["seed"].nunique()
    summary_frame.to_csv(SUMMARY_CSV, index=False)

    print(f"\n=== CONFIG {CONFIG_ID} / {UCB_CONFIG['ucb_config_id']} ===")
    print(f"Completed seeds: {validation_frame['seed'].nunique()}/20")
    print("| Metric | Mean ± Std |")
    print("|---|---:|")
    for row in summary_rows:
        print(f"| {row['metric']} | {row['mean ± std']} |")
    print(f"\nSaved summary: {SUMMARY_CSV}")
